In [0]:
from pyspark.sql import SparkSession, functions as F
from delta import configure_spark_with_delta_pip, DeltaTable

In [0]:
builder = (
    SparkSession.builder.appName("DeltaMergePipeline")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Workspace/Users/yashikumawat53@gmail.com/Sample - Superstore.csv")

In [0]:
new_cols = [c.replace(" ", "_")
              .replace(",", "")
              .replace(";", "")
              .replace("{", "")
              .replace("}", "")
              .replace("(", "")
              .replace(")", "")
              .replace("=", "") for c in df.columns]

df = df.toDF(*new_cols)

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("my_delta_table")
delta_df=spark.table("my_delta_table")
delta_df.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|     Customer_Name|    Segment|      Country|           City|         State|Postal_Code| Region|     Product_ID|       Category|Sub-Category|        Product_Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|       Claire Gute|   Consumer|United States|      Henderson|      Kentucky|      42420|  South|FUR-BO-10001798|   

In [0]:
print("Row count:", delta_df.count())

Row count: 9994


In [0]:
delta_df=delta_df.dropDuplicates(["customer_id"])

In [0]:
delta_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in delta_df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row_ID|Order_ID|Order_Date|Ship_Date|Ship_Mode|Customer_ID|Customer_Name|Segment|Country|City|State|Postal_Code|Region|Product_ID|Category|Sub-Category|Product_Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [0]:
df_2 = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Workspace/Users/yashikumawat53@gmail.com/incremental.csv")

In [0]:
new_cols = [c.replace(" ", "_")
              .replace(",", "")
              .replace(";", "")
              .replace("{", "")
              .replace("}", "")
              .replace("(", "")
              .replace(")", "")
              .replace("=", "") for c in df.columns]

df_2 = df_2.toDF(*new_cols)

In [0]:
df_2.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("incremental_delta_table")
incremental_df=spark.table("incremental_delta_table")
incremental_df.show()

+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+------------+----------+-----------+-------+---------------+---------------+------------+--------------------+--------------------+--------+--------+------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|   Customer_Name|    Segment|      Country|        City|     State|Postal_Code| Region|     Product_ID|       Category|Sub-Category|        Product_Name|               Sales|Quantity|Discount|Profit|
+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+------------+----------+-----------+-------+---------------+---------------+------------+--------------------+--------------------+--------+--------+------+
|  9995|CA-2020-990001|2016-12-21|2016-12-25|Standard Class|   AP-10720|      Anne Pryor|Home Office|United States| Los Angeles|California|      90032|   West|OFF-AR-10004752|Office Su

In [0]:
from delta.tables import DeltaTable
delta_table = DeltaTable.forName(spark, "my_delta_table")

In [0]:

(
    delta_table.alias("target")
    .merge(
        incremental_df.alias("source"),
        "target.Row_ID = source.Row_ID"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
print("Total records after merge:", spark.table("my_delta_table").count())

Total records after merge: 10014


In [0]:
spark.table("my_delta_table").show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|     Customer_Name|    Segment|      Country|           City|         State|Postal_Code| Region|     Product_ID|       Category|Sub-Category|        Product_Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|       Claire Gute|   Consumer|United States|      Henderson|      Kentucky|      42420|  South|FUR-BO-10001798|   